# Lab 8: Human Activity Recognition

**Task:** recognise what a person is doing from the body-worn motion sensors they carry.

**Dataset:** OPPORTUNITY Activity Recognition (UCI: https://archive.ics.uci.edu/dataset/226/opportunity+activity+recognition).

> **The data is not in this repository: the download is about 300 MB (860 MB unzipped), too big for git.** Download the zip from the link above and unzip it into this lab's `data/` folder, so that the files are in `data/OpportunityUCIDataset/dataset/`.

Four people performed morning activities in a kitchen-like room (getting up, making coffee and a sandwich, cleaning up), wearing many sensors: accelerometers and inertial measurement units (IMUs: accelerometer, gyroscope and magnetometer together) on the arms, back, hips and feet. Objects and furniture also had sensors. The data was recorded 30 times a second and labelled by hand on several "tracks": locomotion, high-level activity, and gestures of each arm.

**What we will do:**
1. Read the documentation, load the data, and look at how often each label appears on each track; pick a target with several well-represented classes.
2. Choose and clean the sensor features, and normalise them.
3. Train k-NN (Euclidean vs Manhattan), a linear SVM and an MLP (a simple feedforward neural network).
4. Evaluate with accuracy, per-class precision and recall, and the confusion matrix.
5. Compare the models in a chart and discuss which sensors capture the motion best.

In [ ]:
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
folder = "data/OpportunityUCIDataset/dataset/"
print(open(folder + "../README", encoding="latin-1").read()[:1500] if glob.glob(folder + "../README") else "see the dataset page for documentation")

## 1. Columns, label tracks and response rates

`column_names.txt` lists the 250 columns: time in milliseconds, 242 sensor channels, and 7 label columns. `label_legend.txt` gives the meaning of the label codes.

In [ ]:
names = []
for line in open(folder + "column_names.txt", encoding="latin-1"):
    m = re.match(r"Column: (\d+) (.*)", line.strip())
    if m:
        names.append(m.group(1) + " " + m.group(2).split(";")[0].strip())     # the number keeps names unique
label_cols = names[-7:]
print(names[:4], "...", label_cols)
print(open(folder + "label_legend.txt", encoding="latin-1").read()[:700])

Each subject has five recordings of their normal routine (ADL1–ADL5) and one "drill" run that repeats the activities in a scripted order. Each file holds one row per 1/30 second. We load all of them, keeping every 10th row (3 rows per second): neighbouring rows are almost identical, and this keeps the lab fast.

In [ ]:
frames = []
for path in sorted(glob.glob(folder + "S*-*.dat")):
    subject, run = re.search(r"(S\d)-(\w+)\.dat", path).groups()
    d = pd.read_csv(path, sep=" ", header=None, names=names).iloc[::10]
    d["subject"], d["run"] = subject, run
    frames.append(d)
data = pd.concat(frames, ignore_index=True)
print(data.shape)
print(data.groupby(["subject", "run"]).size().unstack())

### Response rates per label track
How often does each label appear on each track? Code 0 means "no label" (the person was doing none of the labelled things on that track).

In [ ]:
legend = {}
for line in open(folder + "label_legend.txt", encoding="latin-1"):
    parts = [p.strip() for p in line.split("-")]
    if len(parts) >= 3 and parts[0].isdigit():
        legend[int(parts[0])] = parts[-1]
for col in label_cols:
    shares = data[col].value_counts(normalize=True)
    top = ", ".join(f"{legend.get(int(k), 'none' if k == 0 else int(k))} {v:.1%}" for k, v in shares.head(6).items())
    print(f"{col:24s} {data[col].nunique():3d} labels | {top}")

- The **gesture tracks** (left/right arm, objects, both arms) are mostly "none" (78–93% of the time) and split the rest over 10–20 rare gestures: too few examples of each for a first model.
- **HL_Activity** (relaxing, coffee time, early morning, cleanup, sandwich time) covers a whole scene; telling coffee time from sandwich time needs context over minutes, not one instant of sensor readings.
- **Locomotion** has four well-represented classes (stand, walk, sit, lie) plus "none", and should be visible in the body's motion at each instant. **We pick Locomotion** and drop the unlabelled rows.

In [ ]:
loc_names = {1: "Stand", 2: "Walk", 4: "Sit", 5: "Lie"}
data = data[data["244 Locomotion"].isin(loc_names)].copy()
data["activity"] = data["244 Locomotion"].map(loc_names)
print(data["activity"].value_counts().to_dict())

## 2. Features

The sensor channels come in several groups: **accelerometers** and **IMUs** worn on the body, accelerometers attached to **objects** (cup, knife, fridge door...), **reed switches** on doors and drawers, and **location tags** (position in the room). Careful: the object accelerometers are also called "Accelerometer" in the column names, so we select by location. For recognising posture and motion we use the body-worn sensors only: they move with the person, and a model based on them would work in any room. Many channels have gaps (a sensor's wireless connection dropped):

In [ ]:
body_accelerometers = {"RKN^", "HIP", "LUA^", "RUA_", "LH", "BACK", "RKN_", "RWR", "RUA^", "LUA_", "LWR", "RH"}   # from column_names.txt
sensor_cols = [c for c in names[1:-7]
               if c.split()[1] == "InertialMeasurementUnit" or (c.split()[1] == "Accelerometer" and c.split()[2] in body_accelerometers)]
missing = data[sensor_cols].isna().mean()
print(len(sensor_cols), "body-worn channels; missing share: median", f"{missing.median():.1%}", "| channels with > 10% missing:", (missing > 0.1).sum())
keep = missing[missing <= 0.1].index.tolist()
print("kept", len(keep), "channels")

We drop channels missing more than 10% of the time. The remaining gaps are short: we fill them with the last valid value **within the same recording** (a sensor that drops out for a moment usually reads about the same as just before), and any value still missing at the very start of a recording with the training median.

### Train/test split by recording
Rows a few seconds apart are nearly identical. A random split would put near-copies of each test row in the training data and give a misleadingly high score. Instead we train on recordings ADL1–ADL3 and the drill run of all four subjects, and test on the unseen recordings ADL4 and ADL5.

In [ ]:
data[keep] = data.groupby(["subject", "run"])[keep].ffill()
train = data[~data["run"].isin(["ADL4", "ADL5"])]
test = data[data["run"].isin(["ADL4", "ADL5"])]
medians = train[keep].median()
x_train, x_test = train[keep].fillna(medians), test[keep].fillna(medians)
y_train, y_test = train["activity"], test["activity"]
print("train:", x_train.shape, y_train.value_counts().to_dict())
print("test :", x_test.shape, y_test.value_counts().to_dict())

### Normalising
The channels have very different scales: accelerations in milli-g (hundreds to thousands), gyroscope rates, magnetic field values, orientation quaternions between −1 and 1. k-NN and SVMs work with distances or dot products, and neural networks train badly with unscaled inputs, so we standardise every channel (mean 0, standard deviation 1), learning the means and standard deviations from the training data only.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(x_train)
xs_train, xs_test = scaler.transform(x_train), scaler.transform(x_test)

## 3. Models

- **k-NN**: classify each moment by the activities of the $k$ most similar training moments. We compare the **Euclidean** distance ($\sqrt{\sum (x_j - z_j)^2}$, dominated by the largest differences) with the **Manhattan** distance ($\sum |x_j - z_j|$, less sensitive to one channel being far off).
- **Linear SVM**: one maximum-margin hyperplane per class against the rest.
- **MLP**: a feedforward network with two hidden layers of ReLU units, trained with Adam, an L2 penalty and early stopping.

In [ ]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

models = {
    "k-NN, Euclidean (k=15)": KNeighborsClassifier(n_neighbors=15, p=2, n_jobs=-1),
    "k-NN, Manhattan (k=15)": KNeighborsClassifier(n_neighbors=15, p=1, n_jobs=-1),
    "linear SVM": LinearSVC(C=0.1, max_iter=5000),
    "MLP (128, 64)": MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, early_stopping=True, max_iter=200, random_state=0),
}
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder().fit(y_train)     # activity names -> 0, 1, 2, 3 (the MLP's early stopping needs numeric labels)

results, predictions = [], {}
for name, model in models.items():
    t = time.time()
    model.fit(xs_train, encoder.transform(y_train))
    pred = encoder.inverse_transform(model.predict(xs_test))
    predictions[name] = pred
    results.append({"model": name, "accuracy": accuracy_score(y_test, pred), "seconds": time.time() - t})
pd.DataFrame(results).round(3)

The MLP is clearly best (0.88). Manhattan distance does *worse* than Euclidean here (0.79 against 0.83): with 127 standardised channels, many of them noisy, summing absolute differences lets many small irrelevant differences add up, while Euclidean distance is dominated by the few channels that differ a lot (here, posture). Which distance works better is an empirical question. How many neighbours?

In [ ]:
for k in [1, 5, 15, 45]:
    knn = KNeighborsClassifier(n_neighbors=k, p=2, n_jobs=-1).fit(xs_train, y_train)
    print(f"k={k:2d}: test accuracy {accuracy_score(y_test, knn.predict(xs_test)):.3f}")

## 4. Evaluation

Accuracy alone hides which activities are confused. Per-class precision (of the moments predicted as *walk*, how many really were walking) and recall (of the real walking moments, how many were recognised):

In [ ]:
best = max(results, key=lambda r: r["accuracy"])["model"]
print("best model:", best)
print(classification_report(y_test, predictions[best], digits=3))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, predictions[best], normalize="true", values_format=".2f", ax=ax, colorbar=False)
ax.set_title(f"{best}: share of each true activity"); plt.show()

### Comparison chart

In [ ]:
rows = []
for name, pred in predictions.items():
    rep = classification_report(y_test, pred, output_dict=True, zero_division=0)
    rows.append({"model": name, "accuracy": rep["accuracy"], "macro F1": rep["macro avg"]["f1-score"],
                 **{f"recall {c}": rep[c]["recall"] for c in ["Stand", "Walk", "Sit", "Lie"]}})
comparison = pd.DataFrame(rows).set_index("model")
comparison.plot.bar(figsize=(11, 4), rot=0); plt.ylim(0, 1); plt.legend(fontsize=8, ncol=3); plt.title("model comparison on unseen recordings"); plt.show()
comparison.round(3)

Fewer neighbours do better here ($k = 1$: 0.85): walking is a minority of the moments, and with many neighbours the more common standing moments outvote it.

**Which activities are confused most?** *Stand* and *walk* are mixed up most often: k-NN recognises only 36–49% of the walking moments, mostly calling them standing; even the MLP misses about a fifth. At a single instant walking can look like standing (both upright); the difference is the movement *over time*. *Sit* and *lie* are recognised almost perfectly by every model, because the body's orientation is different, which the sensors measure directly.

## 5. Which sensors capture the motion?

A simple way to see which body locations matter: train the same model on each location's sensors alone.

In [ ]:
groups = pd.Series(keep).map(lambda col: col.split()[1][:3].replace("Ine", "IMU") + " " + col.split()[2])   # e.g. "IMU BACK", "Acc HIP"
rows = []
for loc_name in groups.value_counts().index:
    cols = [i for i, g in enumerate(groups) if g == loc_name]
    m = KNeighborsClassifier(n_neighbors=15, p=1, n_jobs=-1).fit(xs_train[:, cols], y_train)
    rows.append({"sensor location": loc_name, "channels": len(cols), "accuracy alone": accuracy_score(y_test, m.predict(xs_test[:, cols]))})
pd.DataFrame(rows).sort_values("accuracy alone", ascending=False).round(3)

### Which features capture motion best? (summary)

- **The IMU on the back is the best single sensor** (0.81 accuracy alone). Posture (stand, sit, lie) is mostly about the **orientation** of the trunk relative to gravity, and an IMU measures orientation directly (its quaternion channels), besides acceleration and rotation. The **right-knee accelerometer** is the best plain accelerometer (0.77): the thigh is horizontal when sitting.
- **IMUs beat plain accelerometers** at almost every location: 13 channels (acceleration, rotation rate, magnetic field, orientation) against 3.
- **The arm IMUs are surprisingly good** (0.76–0.80), even though the arms move all the time in a kitchen: the arm's orientation still reflects whether the person is sitting at a table or lying down.
- **The shoe IMUs are weaker alone** (0.68), although walking is all about the feet. At a single instant, a foot during walking is on the ground half the time and looks like a standing foot; walking shows in the *rhythm* of the foot movements over a second or two, which single instants can't see. The hip accelerometer is the weakest (0.59).

**How to do better:** use **windows** instead of single instants: summarise, say, 2 seconds of each channel by its mean, standard deviation, minimum and maximum (the standard approach in activity recognition). The standard deviation of the foot accelerometers over 2 seconds separates walking from standing far better than any single reading. Try it: group rows into consecutive windows within each recording with `rolling`, compute these statistics, and rerun the models.